# 🎬 Video Dubbing Agent — Colab T4 — Fixed v7

Conversão do HF Space [`sauravghu/video-dubbing-agent`](https://huggingface.co/spaces/sauravghu/video-dubbing-agent) para Colab.

**Como usar:** `Runtime → Run all`. No fim, abra a URL pública impressa pela última célula.

## Correções v7

- Demucs agora salva stems em **WAV**, evitando padding/artefatos de MP3.
- Whisper tem fallback automático em caso de OOM e normaliza `auto`/`pt-br`.
- Tradução tem retries, cache e fallback por segmento.
- Edge-TTS tem retries, divisão de texto longo, ajuste adaptativo de duração e corte com fade se ainda passar do slot.
- Assembly removeu o deslocamento artificial de 20 ms antes da fala, limita cada TTS ao espaço disponível e corrige chunks >10 min.
- Concatenação WAV em vídeos longos agora reencoda PCM corretamente.
- Mix final tem `alimiter`, `dynaudnorm`, `loudnorm -14 LUFS` e valida saída.
- Merge preserva nome original, sufixo de idioma e fallback para reencode se `-c:v copy` falhar.
- Mantidos os controles `VDA_LOG_LEVEL` e `VDA_TUNNEL`.

In [13]:
# @title
# === CÉLULA 1 — Dependências de sistema (ffmpeg) ===
import subprocess, shutil, sys, os

def run(cmd, check=True):
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    if p.stdout:
        print(p.stdout[-800:])
    if p.returncode != 0:
        print("STDERR:", p.stderr[-800:])
        if check:
            raise RuntimeError(f"Command failed: {cmd}")
    return p

if shutil.which("ffmpeg") is None:
    run("apt-get update -qq && apt-get install -y -qq ffmpeg")
else:
    print("ffmpeg already installed.")

run(["ffmpeg", "-version"])
print("✅ Cell 1 OK")

ffmpeg already installed.
libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --enable-libzmq --enable-libzvbi --enable-lv2 --enable-omx --enable-openal --enable-opencl --enable-opengl --enable-sdl2 --enable-pocketsphinx --enable-librsvg --enable-libmfx --enable-libdc1394 --enable-libdrm --enable-libiec61883 --enable-chromaprint --enable-frei0r --enable-libx264 --enable-shared
libavutil      56. 70.100 / 56. 70.100
libavcodec     58.134.100 / 58.134.100
libavformat    58. 76.100 / 58. 76.100
libavdevice    58. 13.100 / 58. 13.100
libavfilter     7.110.100 /  7.110.100
libswscale      5.  9.100 /  5.  9.100
libswresample   3.  9.100 /  3.  9.100
libpostproc    55.  9.100 / 55.  9.100

✅ Cell 1 OK


In [14]:
# @title
# === CÉLULA 2 — Clonar o HF Space ===
import os, sys, subprocess

try:
    import huggingface_hub  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub==0.23.4"], check=True)

from huggingface_hub import snapshot_download

REPO_ID = "sauravghu/video-dubbing-agent"
_BASE = "/content" if os.path.isdir("/content") and os.access("/content", os.W_OK) else os.path.expanduser("~")
APP_DIR = os.path.join(_BASE, "video-dubbing-agent")
os.environ["VDA_APP_DIR"] = APP_DIR

if not os.path.isdir(APP_DIR) or not os.listdir(APP_DIR):
    try:
        local = snapshot_download(repo_id=REPO_ID, repo_type="space", local_dir=APP_DIR)
        print("Cloned via snapshot_download:", local)
    except Exception as e:
        print("snapshot_download failed:", e, "— falling back to git+GIT_LFS_SKIP_SMUDGE")
        os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
        if os.path.isdir(APP_DIR):
            subprocess.run(["rm", "-rf", APP_DIR], check=False)
        subprocess.run(["git", "clone", f"https://huggingface.co/spaces/{REPO_ID}", APP_DIR], check=True)

assert os.path.isdir(APP_DIR), "Repo directory missing"
print("Files in repo:")
for f in sorted(os.listdir(APP_DIR)):
    print(" ", f)
assert "main.py" in os.listdir(APP_DIR), "main.py missing — repo clone incomplete"

if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

os.chdir(APP_DIR)
print("CWD =", os.getcwd())
print("✅ Cell 2 OK")

Files in repo:
  .cache
  .gitattributes
  Dockerfile
  README.md
  __pycache__
  config.py
  jobs
  main.py
  requirements.txt
  routes
  services
  static
  temp_jobs
  templates
  utils
CWD = /content/video-dubbing-agent
✅ Cell 2 OK


In [15]:
# @title
# === CÉLULA 3 — Instalação Python (constraints + sequência segura) ===
import sys, subprocess, importlib, os, shutil, urllib.request, stat

_BASE = "/content" if os.path.isdir("/content") and os.access("/content", os.W_OK) else os.path.expanduser("~")
CONSTRAINTS = os.path.join(_BASE, "constraints.txt")
with open(CONSTRAINTS, "w") as f:
    f.write("numpy==1.26.4\n")
    f.write("huggingface_hub==0.23.4\n")

def pip(*args, force=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--constraint", CONSTRAINTS, *args]
    print(">>", " ".join(cmd[-min(len(cmd), 8):]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("STDOUT:", r.stdout[-1500:])
        print("STDERR:", r.stderr[-1500:])
        if not force:
            raise RuntimeError("pip install failed")
    return r

print("Step A — numpy 1.26.4 (forced, no-deps)")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", "numpy==1.26.4"],
    check=True,
)

print("Step B — huggingface_hub 0.23.4 (pinned)")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", "huggingface_hub==0.23.4"],
    check=True,
)

print("Step C — core web stack compatível com anyio 4")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "fastapi>=0.115,<0.117",
        "starlette>=0.41,<0.42",
        "uvicorn[standard]>=0.30,<0.35",
        "anyio>=4.4,<5",
        "python-multipart>=0.0.6",
        "pydantic>=2.5,<3",
    ],
    check=True,
)

print("Step C2 — utilitários")
pip(
    "yt-dlp",
    "deep-translator==1.11.4",
    "edge-tts>=6.1.9",
    "pydub==0.25.1",
    "requests>=2.31.0",
)

print("Step D — ASR + audio (GPU-aware)")
_has_nvidia = shutil.which("nvidia-smi") is not None
if _has_nvidia:
    print("   GPU detectada → ctranslate2 4.5.0 (CUDA 12)")
    pip("ctranslate2==4.5.0", "faster-whisper==1.0.3", "soundfile", "av")
else:
    print("   sem GPU → ctranslate2 4.4.0")
    pip("ctranslate2==4.4.0", "faster-whisper==1.0.3", "soundfile", "av")

try:
    pip("pyngrok", force=True)
except Exception:
    pass

print("Step E — demucs opcional")
try:
    pip("demucs==4.0.1")
except Exception as e:
    print("demucs install failed (OK — pipeline falls back):", e)

print("Step F — cloudflared binary oficial")
CF_BIN = "/usr/local/bin/cloudflared"
if not os.path.exists(CF_BIN):
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, CF_BIN)
    os.chmod(CF_BIN, os.stat(CF_BIN).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
subprocess.run([CF_BIN, "--version"], check=True)

print("Step G — nest_asyncio")
pip("nest_asyncio")

print("Step H — purging sys.modules caches")
_PURGE_PREFIXES = (
    "numpy", "pandas", "huggingface_hub",
    "ctranslate2", "faster_whisper",
    "anyio", "fastapi", "starlette", "uvicorn",
    "h11", "httpcore", "httpx",
    "av", "soundfile", "demucs",
)
for m in list(sys.modules):
    if any(m == p or m.startswith(p + ".") for p in _PURGE_PREFIXES):
        sys.modules.pop(m, None)
import gc as _gc; _gc.collect(); _gc.collect(); _gc.collect(); del _gc
importlib.invalidate_caches()
# Força re-validação na Cell 3b após esta purga
os.environ.pop("_VDA_ENV_HARDENED_V1", None)

import numpy as _np
print("numpy:", _np.__version__)
assert _np.__version__ == "1.26.4", f"numpy ABI not pinned ({_np.__version__})"

import huggingface_hub as _hh
print("huggingface_hub:", _hh.__version__)
print("✅ Cell 3 OK — pip install complete")

Step A — numpy 1.26.4 (forced, no-deps)
Step B — huggingface_hub 0.23.4 (pinned)
Step C — core web stack compatível com anyio 4
Step C2 — utilitários
>> -q --constraint /content/constraints.txt yt-dlp deep-translator==1.11.4 edge-tts>=6.1.9 pydub==0.25.1 requests>=2.31.0
Step D — ASR + audio (GPU-aware)
   GPU detectada → ctranslate2 4.5.0 (CUDA 12)
>> install -q --constraint /content/constraints.txt ctranslate2==4.5.0 faster-whisper==1.0.3 soundfile av
>> /usr/bin/python3 -m pip install -q --constraint /content/constraints.txt pyngrok
Step E — demucs opcional
>> /usr/bin/python3 -m pip install -q --constraint /content/constraints.txt demucs==4.0.1
Step F — cloudflared binary oficial
Step G — nest_asyncio
>> /usr/bin/python3 -m pip install -q --constraint /content/constraints.txt nest_asyncio
Step H — purging sys.modules caches
numpy: 1.26.4
huggingface_hub: 0.23.4
✅ Cell 3 OK — pip install complete


/tmp/ipykernel_3502/388222342.py:104: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  import numpy as _np


In [16]:
# @title
# === CÉLULA 3b — Garantia de Integridade AI ===
# Estratégia: torch NUNCA é purgado (não é reinstalado em nenhuma etapa).
# Step H purga apenas ctranslate2/faster-whisper (que SÃO reinstalados em Step D).
# Sem reimport de torch → sem colisão de docstring → sem memory_format error.

import gc, importlib, os, sys

_SENTINEL = "_VDA_ENV_HARDENED_V1"


def _torch_ok():
    """torch está em sys.modules e completamente inicializado."""
    t = sys.modules.get("torch")
    return (
        t is not None
        and hasattr(t, "memory_format")
        and hasattr(t, "Tensor")
        and hasattr(t, "cuda")
    )


def _ensure_torch():
    """
    Garante que torch está carregado.
    Só chega aqui num kernel recém-iniciado (torch não foi pré-carregado pelo Colab).
    Como ml_doc ainda não está gravado, o import é limpo — sem colisão possível.
    """
    if _torch_ok():
        return
    # Remove qualquer estado parcial antes do import limpo
    for m in list(sys.modules):
        if m == "torch" or m.startswith("torch."):
            sys.modules.pop(m, None)
    gc.collect(); gc.collect(); importlib.invalidate_caches()
    import torch  # noqa  — primeiro load no kernel, sem colisão


def _ensure_ctranslate2_whisper():
    """
    Purga e reimporta ctranslate2 + faster_whisper.
    Step H já limpou sys.modules desses pacotes (foram reinstalados em Step D),
    então este import sempre pega a versão recém-instalada.
    torch permanece em sys.modules intocado — nenhum reimport seu ocorre.
    """
    for m in list(sys.modules):
        if (m == "ctranslate2" or m.startswith("ctranslate2.")
                or m == "faster_whisper" or m.startswith("faster_whisper.")):
            sys.modules.pop(m, None)
    gc.collect(); importlib.invalidate_caches()
    import ctranslate2     # noqa
    import faster_whisper  # noqa
    return (
        f"faster_whisper {getattr(faster_whisper, '__version__', '?')}"
        f" / ctranslate2 {ctranslate2.__version__}"
    )


# ── MAIN (idempotente por sentinel + verificação funcional) ──────────────────

_already_ok = (
    os.environ.get(_SENTINEL) == "1"
    and _torch_ok()
    and "faster_whisper" in sys.modules
)

if _already_ok:
    print("✅ Cell 3b: ambiente íntegro — pulando.")
else:
    os.environ.pop(_SENTINEL, None)
    print("🔒 Cell 3b — verificando integridade AI...")

    # 1. Torch
    try:
        _ensure_torch()
        import torch as _t
        print(f"   [1] torch {_t.__version__}: ✅")
        del _t
    except Exception as _e:
        print(f"   [1] torch: ❌ {_e}")
        raise SystemExit("Torch falhou. Runtime → Restart runtime → Run all")

    # 2. ctranslate2 + faster_whisper
    try:
        _msg = _ensure_ctranslate2_whisper()
        print(f"   [2] {_msg}: ✅")
    except Exception as _e:
        print(f"   [2] ctranslate2/faster_whisper: ❌ {_e}")
        raise SystemExit(f"Dependência AI falhou: {_e}")

    # 3. Relatório GPU (informativo)
    try:
        import torch as _t, ctranslate2 as _ct
        _has_gpu = _t.cuda.is_available()
        _ct_n = _ct.get_cuda_device_count() if _has_gpu else 0
        print(f"   [3] GPU: {'✅ CUDA ' + _t.cuda.get_device_name(0) if _has_gpu else '⚠️  CPU'}"
              f" | ctranslate2 devices: {_ct_n}")
        del _t, _ct
    except Exception:
        pass

    os.environ[_SENTINEL] = "1"
    print("\n✅ Cell 3b OK — ambiente garantido.")

🔒 Cell 3b — verificando integridade AI...
   [1] torch 2.11.0+cu128: ✅
   [2] faster_whisper 1.0.3 / ctranslate2 4.5.0: ✅
   [3] GPU: ✅ CUDA Tesla T4 | ctranslate2 devices: 1

✅ Cell 3b OK — ambiente garantido.


In [17]:
# @title
# === CÉLULA 4 — Patches defensivos + imports do app ===
import sys, os, glob, importlib, types, subprocess

def _pkg_ver(name):
    import importlib.metadata as _md
    try:
        return _md.version(name)
    except Exception:
        return "0.0.0"

def _ver_ok():
    try:
        return (
            int(_pkg_ver("anyio").split(".")[0]) >= 4
            and int(_pkg_ver("starlette").split(".")[1]) >= 41
            and float(_pkg_ver("fastapi").rsplit(".", 1)[0]) >= 0.115
        )
    except Exception:
        return False

if not _ver_ok():
    print("Web stack desatualizada — atualizando agora sem constraints...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--upgrade",
            "fastapi>=0.115,<0.117",
            "starlette>=0.41,<0.42",
            "uvicorn[standard]>=0.30,<0.35",
            "anyio>=4.4,<5",
        ],
        check=False,
    )

for _m in list(sys.modules):
    if (
        _m == "anyio" or _m.startswith("anyio.")
        or _m == "starlette" or _m.startswith("starlette.")
        or _m == "fastapi" or _m.startswith("fastapi.")
        or _m == "uvicorn" or _m.startswith("uvicorn.")
        or _m == "h11" or _m.startswith("h11.")
        or _m == "httpcore" or _m.startswith("httpcore.")
    ):
        sys.modules.pop(_m, None)
importlib.invalidate_caches()

print(
    "anyio:", _pkg_ver("anyio"),
    "| fastapi:", _pkg_ver("fastapi"),
    "| starlette:", _pkg_ver("starlette"),
    "| uvicorn:", _pkg_ver("uvicorn"),
)
assert int(_pkg_ver("anyio").split(".")[0]) >= 4, "anyio must be >=4"
assert int(_pkg_ver("starlette").split(".")[1]) >= 41, "starlette must be >=0.41"

import huggingface_hub as hh
if not hasattr(hh, "HfFolder"):
    class _HfFolder:
        @staticmethod
        def get_token():
            return os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")

        @staticmethod
        def save_token(t):
            os.environ["HF_TOKEN"] = t

    hh.HfFolder = _HfFolder
    print("Patched HfFolder shim")

try:
    import torch
    if not getattr(torch.load, "_vda_patched", False):
        _orig_load = torch.load
        def _safe_load(*a, **kw):
            kw.setdefault("weights_only", False)
            return _orig_load(*a, **kw)
        _safe_load._vda_patched = True
        torch.load = _safe_load
        print("Patched torch.load (weights_only=False default)")
    else:
        print("torch.load already patched — skipping")
except Exception:
    pass

if "spaces" not in sys.modules:
    spaces = types.ModuleType("spaces")

    def _gpu(*a, **kw):
        if a and callable(a[0]):
            return a[0]

        def _dec(f):
            return f

        return _dec

    spaces.GPU = _gpu
    sys.modules["spaces"] = spaces

def _apply_coqpit_patch():
    for fp in glob.glob("/usr/local/lib/python3*/dist-packages/coqpit/coqpit.py"):
        try:
            src = open(fp).read()
        except Exception:
            continue
        if "_COLAB_PATCHED_V3" in src:
            return
        src = src.replace(
            "if issubclass(field_type, Serializable):",
            "if isinstance(field_type, type) and issubclass(field_type, Serializable):",
        )
        OLD = "def _deserialize(x, field_type):\n"
        NEW = (
            "def _deserialize(x, field_type):  # _COLAB_PATCHED_V3\n"
            "    import types as _pct, typing as _pty\n"
            "    _UT = getattr(_pct, 'UnionType', None)\n"
            "    if _UT and isinstance(field_type, _UT):\n"
            "        for _t in [t for t in _pty.get_args(field_type) if t is not type(None)]:\n"
            "            try: return _deserialize(x, _t)\n"
            "            except: pass\n"
            "        return x\n"
            "    if getattr(field_type, '__origin__', None) is _pty.Union:\n"
            "        for _t in [t for t in _pty.get_args(field_type) if t is not type(None)]:\n"
            "            try: return _deserialize(x, _t)\n"
            "            except: pass\n"
            "        return x\n"
            "    if not isinstance(field_type, type): return x\n"
        )
        try:
            open(fp, "w").write(src.replace(OLD, NEW, 1))
        except Exception:
            pass
        for m in list(sys.modules):
            if m == "coqpit" or m.startswith("coqpit.") or m == "TTS" or m.startswith("TTS."):
                sys.modules.pop(m, None)
        importlib.invalidate_caches()

_apply_coqpit_patch()

APP_DIR = os.environ.get("VDA_APP_DIR", "/content/video-dubbing-agent")
os.chdir(APP_DIR)
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

for mod in [
    "config", "jobs.progress_tracker", "jobs.pipeline",
    "services.downloader", "services.audio_extractor", "services.vocal_separator",
    "services.transcriber", "services.speaker_profiler", "services.translator",
    "services.tts_generator", "services.audio_assembler", "services.audio_mixer",
    "services.merger", "services.subtitle_generator", "routes.dub", "routes.info",
    "utils.file_manager", "main",
]:
    importlib.import_module(mod)

import faster_whisper, edge_tts, deep_translator, fastapi, uvicorn, yt_dlp
print("faster_whisper:", getattr(faster_whisper, "__version__", "?"))
print("fastapi:", fastapi.__version__)
print("uvicorn:", uvicorn.__version__)
print("✅ Cell 4 OK — all imports successful")

anyio: 4.13.0 | fastapi: 0.116.2 | starlette: 0.41.3 | uvicorn: 0.34.3
torch.load already patched — skipping
faster_whisper: 1.0.3
fastapi: 0.116.2
uvicorn: 0.34.3
✅ Cell 4 OK — all imports successful


In [19]:
# @title
# === CELULA 4b — Patches de Performance ULTRA v7 ===
import os, sys, importlib, subprocess, textwrap, json, re
from pathlib import Path

APP_DIR = os.environ.get("VDA_APP_DIR", "/content/video-dubbing-agent")
os.chdir(APP_DIR)

# --- Configuracoes de Alta Performance ---
WHISPER_MODEL = "large-v3"
BATCH_SIZE = 24
TTS_CONCURRENCY = 12
DEVICE = "cuda"
COMPUTE = "float16"

print(f"Otimizando para Tesla T4: Whisper Batch={BATCH_SIZE}, TTS Concurrency={TTS_CONCURRENCY}")

def _write(rel, content):
    p = Path(rel)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")

# Patch Transcriber com Batch Agressivo
_write("services/transcriber.py", f"""
import logging, subprocess, json, gc, os
from pathlib import Path
from faster_whisper import WhisperModel, BatchedInferencePipeline

MODEL_SIZE = '{WHISPER_MODEL}'
BATCH_SIZE = {BATCH_SIZE}

_cache = {{}}

def transcribe_audio(audio_path, output_dir, source_language=None, device='cuda', progress_callback=None):
    if 'pipe' not in _cache:
        model = WhisperModel(MODEL_SIZE, device='cuda', compute_type='float16')
        _cache['pipe'] = BatchedInferencePipeline(model=model)

    pipe = _cache['pipe']
    segments, info = pipe.transcribe(
        str(audio_path),
        batch_size=BATCH_SIZE,
        language=None if source_language in ['auto', None] else source_language,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=400)
    )

    res = []
    for s in segments:
        res.append({{'start': s.start, 'end': s.end, 'text': s.text, 'speaker': 'SPEAKER_00'}})

    payload = {{'language': info.language, 'segments': res}}
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    (Path(output_dir) / 'transcript.json').write_text(json.dumps(payload))
    return res
""")

# Patch TTS com maior concorrencia
os.environ["VDA_TTS_CONCURRENCY"] = str(TTS_CONCURRENCY)

print("\n✅ Patches de performance aplicados. Reinicie a Celula 7.")

Otimizando para Tesla T4: Whisper Batch=24, TTS Concurrency=12

✅ Patches de performance aplicados. Reinicie a Celula 7.


In [20]:
# @title
# === CÉLULA 5 — Validação leve do ambiente ===
import gc, os, subprocess, sys

print("=== Ambiente ===")
try:
    import torch
    if torch.cuda.is_available():
        print(f"✅ CUDA: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM total: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
        print(f"   PyTorch: {torch.__version__}")
    else:
        print("⚠️ Sem GPU — o app vai usar CPU, bem mais lento")
except Exception as e:
    print("PyTorch erro:", e)

try:
    import ctranslate2
    devices = ctranslate2.get_cuda_device_count()
    print(f"✅ ctranslate2: {ctranslate2.__version__}, CUDA devices: {devices}")
except Exception as e:
    print("ctranslate2 erro:", e)

try:
    import faster_whisper
    print(f"✅ faster-whisper: {faster_whisper.__version__}")
except Exception as e:
    print("faster-whisper erro:", e)

r = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print("✅ ffmpeg:", r.stdout.split("\n")[0])

print("\n✅ Cell 5 OK — ambiente pronto.")
print("   O modelo Whisper será baixado automaticamente no primeiro job real e ficará em cache.")

=== Ambiente ===
✅ CUDA: Tesla T4
   VRAM total: 15.6 GB
   PyTorch: 2.11.0+cu128
✅ ctranslate2: 4.5.0, CUDA devices: 1
✅ faster-whisper: 1.0.3
✅ ffmpeg: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers

✅ Cell 5 OK — ambiente pronto.
   O modelo Whisper será baixado automaticamente no primeiro job real e ficará em cache.


In [25]:
# @title
# === CÉLULA 6 — Smoke test + Recuperação de Integridade ===
import os, sys, importlib, gc, subprocess

APP_DIR = os.environ.get("VDA_APP_DIR", "/content/video-dubbing-agent")
os.chdir(APP_DIR)
if APP_DIR not in sys.path: sys.path.insert(0, APP_DIR)

# --- CICLO DE TESTE-AJUSTE ---
def check_and_fix_imports():
    # 1. Limpeza de Cache de Módulos
    to_purge = ["services", "jobs", "routes", "faster_whisper", "ctranslate2"]
    for m in list(sys.modules):
        if any(p in m for p in to_purge):
            sys.modules.pop(m, None)

    gc.collect()
    importlib.invalidate_caches()

    has_ultra = False
    try:
        from faster_whisper import BatchedInferencePipeline
        print("✅ Ultra-Performance (Batched) detectado.")
        has_ultra = True
    except ImportError:
        print("ℹ️ BatchedInferencePipeline não disponível no path. Usando Modo Estável.")
        # Patch de emergência no arquivo para não dar erro de import
        t_path = "services/transcriber.py"
        if os.path.exists(t_path):
            src = open(t_path).read()
            if "BatchedInferencePipeline" in src:
                src = src.replace("from faster_whisper import WhisperModel, BatchedInferencePipeline", "from faster_whisper import WhisperModel")
                src = src.replace("_cache['pipe'] = BatchedInferencePipeline(model=model)", "_cache['pipe'] = model")
                src = src.replace("batch_size=BATCH_SIZE,", "")
                open(t_path, "w").write(src)
                print("🔧 Patch revertido para Modo Estável para evitar crash.")

    # 2. Importação Final do App
    try:
        importlib.invalidate_caches()
        from services import transcriber, vocal_separator, translator, tts_generator
        from jobs import pipeline
        print("✅ Módulos do Agente carregados com sucesso.")
        return True
    except Exception as e:
        print(f"❌ Falha crítica no carregamento: {e}")
        return False

if check_and_fix_imports():
    print("\n🚀 TUDO PRONTO. Pode iniciar a Célula 7.")
else:
    print("\n❌ Ocorreu um erro. Tente reiniciar o ambiente em Runtime -> Restart session.")

ℹ️ BatchedInferencePipeline não disponível no path. Usando Modo Estável.
🔧 Patch revertido para Modo Estável para evitar crash.
✅ Módulos do Agente carregados com sucesso.

🚀 TUDO PRONTO. Pode iniciar a Célula 7.


In [ ]:
# @title CÉLULA 7 — Iniciar servidor FastAPI + URL pública
import os, sys, threading, time, socket, logging, datetime, subprocess, re, shutil
import urllib.request as _req

# =============================================================================
# 🛠️ CONFIGURAÇÃO DE LOGS (DEBUG / MONITORAMENTO)
# =============================================================================
# Você pode alterar o nível de detalhes das mensagens no console mudando
# a variável abaixo. Opções disponíveis:
# - "DEBUG": Mostra TUDO (detalhes técnicos, comandos ffmpeg, payloads JSON).
# - "INFO":  (Padrão) Mostra o progresso das etapas e status do servidor.
# - "WARNING": Mostra apenas avisos e erros.
# - "ERROR":   Mostra apenas falhas críticas.
#
# Exemplo para desenvolvedores: os.environ["VDA_LOG_LEVEL"] = "DEBUG"
# =============================================================================
LOG_LEVEL = os.environ.get("VDA_LOG_LEVEL", "INFO").upper()
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s %(levelname)s [%(name)s] %(message)s", force=True)

APP_DIR = os.environ.get("VDA_APP_DIR", "/content/video-dubbing-agent")
os.chdir(APP_DIR)
if APP_DIR not in sys.path: sys.path.insert(0, APP_DIR)

import nest_asyncio
nest_asyncio.apply()

import uvicorn
from main import app

PORT = 7860
def _free_port(p):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try: return s.connect_ex(("127.0.0.1", p)) != 0
    finally: s.close()

# Inicia o servidor se a porta estiver livre
if _free_port(PORT):
    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info", loop="asyncio")
    server = uvicorn.Server(config)
    threading.Thread(target=server.run, daemon=True).start()
    print("Aguardando servidor iniciar...")
    time.sleep(5)

# --- Gerenciamento de Túnel (Cloudflare First) ---
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
CF_BIN = "/usr/local/bin/cloudflared"
CF_LOG = "/tmp/cloudflared.log"
open(CF_LOG, "w").close()

print("[tunnel] Iniciando Cloudflare...")
p_cf = subprocess.Popen([
    CF_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
    "--no-autoupdate", "--logfile", CF_LOG
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

PUBLIC_URL = None
for _ in range(30):
    time.sleep(1)
    try:
        log = open(CF_LOG).read()
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", log)
        if m:
            PUBLIC_URL = m.group(0)
            break
    except: pass

if PUBLIC_URL:
    print("\n" + "="*60)
    print(f"🌐 URL PÚBLICA: {PUBLIC_URL}")
    print("="*60)
else:
    print("❌ Falha ao obter URL do Cloudflare. Verifique /tmp/cloudflared.log")

try:
    while True:
        time.sleep(30)
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] ✅ Servidor Ativo | {PUBLIC_URL}")
except KeyboardInterrupt:
    print("Encerrando...")

[tunnel] Iniciando Cloudflare...

🌐 URL PÚBLICA: https://etc-sessions-certification-pulled.trycloudflare.com
INFO:     189.124.147.9:0 - "GET / HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK


2026-05-28 23:58:29,519 WARNING [vda.http] RESP GET /favicon.ico -> 404 0.5ms type=application/json


INFO:     189.124.147.9:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
[23:58:47] ✅ Servidor Ativo | https://etc-sessions-certification-pulled.trycloudflare.com


2026-05-28 23:59:10,739 INFO [utils.file_manager] Created job directory: /content/video-dubbing-agent/temp_jobs/903b6439
2026-05-28 23:59:10,781 INFO [routes.dub] Upload received: Gvc03610OZhxUgjH.mp4 (56.8 MB) → /content/video-dubbing-agent/temp_jobs/903b6439/Gvc03610OZhxUgjH.mp4
2026-05-28 23:59:10,783 INFO [jobs.pipeline] === Pipeline started: job=903b6439, target=pt ===
2026-05-28 23:59:10,785 INFO [jobs.pipeline] STAGE 1: Downloading video...
2026-05-28 23:59:10,785 INFO [services.downloader] File already in job dir: /content/video-dubbing-agent/temp_jobs/903b6439/Gvc03610OZhxUgjH.mp4
2026-05-28 23:59:10,786 INFO [jobs.pipeline] STAGE 2: Extracting audio...
2026-05-28 23:59:10,787 INFO [services.audio_extractor] Extracting audio for STT (16kHz mono WAV)...


INFO:     189.124.147.9:0 - "POST /api/dub-upload HTTP/1.1" 200 OK


...

INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK
[00:01:47] ✅ Servidor Ativo | https://etc-sessions-certification-pulled.trycloudflare.com
INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 

2026-05-29 00:02:02,052 INFO [jobs.pipeline] STAGE 10: Merging dubbed audio into video...


INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK


2026-05-29 00:02:12,357 INFO [services.merger] Final video: /content/video-dubbing-agent/temp_jobs/903b6439/Gvc03610OZhxUgjH__dub.mp4
2026-05-29 00:02:12,358 INFO [jobs.pipeline] STAGE 11: Generating subtitles...
2026-05-29 00:02:12,360 INFO [services.subtitle_generator] Original subtitles: /content/video-dubbing-agent/temp_jobs/903b6439/subtitles_en.srt
2026-05-29 00:02:12,361 INFO [services.subtitle_generator] Translated subtitles: /content/video-dubbing-agent/temp_jobs/903b6439/subtitles_pt.srt
2026-05-29 00:02:12,362 INFO [jobs.pipeline] === Pipeline complete: /content/video-dubbing-agent/temp_jobs/903b6439/Gvc03610OZhxUgjH__dub.mp4 ===


INFO:     189.124.147.9:0 - "GET /api/status/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/jobs HTTP/1.1" 200 OK
[00:02:17] ✅ Servidor Ativo | https://etc-sessions-certification-pulled.trycloudflare.com
[00:02:47] ✅ Servidor Ativo | https://etc-sessions-certification-pulled.trycloudflare.com
INFO:     189.124.147.9:0 - "GET /api/download/903b6439 HTTP/1.1" 200 OK
INFO:     189.124.147.9:0 - "GET /api/download/903b6439 HTTP/1.1" 206 Partial Content
[00:03:17] ✅ Servidor Ativo | https://etc-sessions-certification-pulled.trycloudflare.com
[00:03:47] ✅ Servidor Ativo | https://etc-sessions-certification-pulled.trycloudflare.com
[00:04:17] ✅ Servidor Ativo | https://etc-sessions-certification-pulled.trycloudflare.com
[00:04:47] ✅ Servidor Ativo | https://etc-sessions-certification-pulled.trycloudflare.com
[00:05:17] ✅ Servidor Ativo | https://etc-sessions-certification-pulled.trycloudflare.com
[00:05:47] ✅ Servidor Ativo | https://etc-sessions-certification-pulled.trycloud

### 🚀 Download de Alta Velocidade (Bypass de Túnel)
Use esta célula se o download pela interface web estiver lento. Ela utiliza o sistema direto do Google Colab para transferir o arquivo.

In [ ]:
import os
from google.colab import files
from pathlib import Path

# Procura por arquivos MP4 nas pastas de jobs
base_path = Path('/content/video-dubbing-agent/temp_jobs')
dubbed_files = list(base_path.glob('**/output/*.mp4'))

if not dubbed_files:
    print("❌ Nenhum vídeo dublado encontrado ainda. Certifique-se de que o processo terminou.")
else:
    print("✅ Arquivos encontrados:")
    for i, path in enumerate(dubbed_files):
        print(f"[{i}] {path.name} ({path.stat().st_size / (1024*1024):.2f} MB)")

    # Escolha o índice do arquivo que deseja baixar (ex: 0)
    idx = 0
    target = dubbed_files[idx]
    print(f"\nIniciando download direto de: {target.name}...")
    files.download(str(target))